# PSSP benchmark — real training & evaluation (Colab)

This notebook **actually trains and evaluates** the five models described in the manuscript on **real, public data** and writes every number the paper needs to `results_real/`. Nothing is hard-coded: every metric, loss curve, p-value, parameter count and timing is computed in this run.

**How to run**
1. `Runtime → Change runtime type → T4 GPU` (any GPU works).
2. First set `QUICK_TEST = True` in the config cell and `Runtime → Run all` (≈5–10 min) to check that everything works.
3. Then set `QUICK_TEST = False`, `Runtime → Restart session and run all` (≈1–2 h on a T4).
4. At the end a file `results_real.zip` is downloaded. Send it back for the manuscript update.

**Data**: NetSurfP-2.0 training set and the CB513 / TS115 / CASP12 / CASP13 / CASP14 test sets, as distributed in the Hugging Face dataset `proteinea/secondary_structure_prediction` (DOI 10.57967/hf/1104). Training chains with ≥25% identity (≥50% coverage) to any test chain are removed with MMseqs2 before training.

In [ ]:
# ==== 1. Configuration ====
QUICK_TEST   = True          # True: tiny smoke test (NOT for the paper). False: full run.
SEEDS        = [42, 123, 2026]
SPLIT_SEED   = 0             # fixed train/validation split, identical for all models and seeds
VAL_FRACTION = 0.10
FILTER_LEAKAGE = True        # MMseqs2 filter of training chains vs. all test sets
LEAK_MIN_ID  = 0.25
LEAK_MIN_COV = 0.50
TEST_SETS    = ['CB513', 'TS115', 'CASP12', 'CASP13', 'CASP14']
STATS_TEST_SET = 'CB513'     # set used for McNemar / Wilcoxon / bootstrap
STATS_SEED   = 42            # seed whose predictions are used for the paired tests
BOOT_B       = 10000
N_BENCH_GPU  = 100           # sequences used for GPU throughput (batch size 1)
N_BENCH_CPU  = 50            # sequences used for CPU throughput (batch size 1)
CPU_THREADS  = 2
SAVE_TO_DRIVE = False        # True: also copy results to Google Drive
OUT = '/content/results_real'

CFG = {   # exact hyper-parameters actually used (copy these into the paper)
  'WindowMLP':             dict(kind='seq', opt='Adam',  lr=1e-3, wd=1e-4, batch=32, max_epochs=20, patience=3),
  'Hybrid 1D-CNN-BiLSTM':  dict(kind='seq', opt='AdamW', lr=1e-3, wd=1e-4, batch=16, max_epochs=20, patience=3),
  'ESM-2 (8M) Linear Probe': dict(kind='head', backbone='facebook/esm2_t6_8M_UR50D',   head='linear', opt='Adam', lr=1e-3, wd=0.0, batch=4096, max_epochs=30, patience=4),
  'ESM-2 (8M) MLP Head':     dict(kind='head', backbone='facebook/esm2_t6_8M_UR50D',   head='mlp',    opt='Adam', lr=1e-3, wd=0.0, batch=4096, max_epochs=30, patience=4),
  'ESM-2 (35M) MLP Head':    dict(kind='head', backbone='facebook/esm2_t12_35M_UR50D', head='mlp',    opt='Adam', lr=1e-3, wd=0.0, batch=4096, max_epochs=30, patience=4),
}
if QUICK_TEST:
    SEEDS = [42]; STATS_SEED = 42; BOOT_B = 1000; N_BENCH_GPU = 10; N_BENCH_CPU = 5
    for c in CFG.values(): c['max_epochs'] = 2
print('QUICK_TEST =', QUICK_TEST)

In [ ]:
# ==== 2. Environment ====
!pip -q install transformers
import os, sys, json, time, copy, random, platform, subprocess, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import transformers
os.makedirs(OUT, exist_ok=True)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert dev.type == 'cuda', 'Select a GPU runtime (Runtime -> Change runtime type -> T4 GPU)'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
cpu_model = subprocess.run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'", shell=True, capture_output=True, text=True).stdout.strip()
ENV = dict(gpu=torch.cuda.get_device_name(0), cpu=cpu_model, cpu_threads_used=CPU_THREADS, n_cpu=os.cpu_count(),
           torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version(),
           cuda=torch.version.cuda, quick_test=QUICK_TEST, date=time.strftime('%Y-%m-%d %H:%M'))
print(json.dumps(ENV, indent=2))
json.dump(ENV, open(f'{OUT}/environment.json', 'w'), indent=2)
json.dump(CFG, open(f'{OUT}/hyperparameters.json', 'w'), indent=2)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ==== 3. Download and parse real data ====
BASE = 'https://huggingface.co/datasets/proteinea/secondary_structure_prediction/resolve/main/'
FILES = {'TRAIN': 'training_hhblits.csv', **{t: f'{t}.csv' for t in TEST_SETS}}
os.makedirs('/content/data', exist_ok=True)
for k, f in FILES.items():
    p = f'/content/data/{f}'
    if not os.path.exists(p):
        urllib.request.urlretrieve(BASE + f, p)
    print(k, f, os.path.getsize(p) // 1024, 'KB')

AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_IX = {a: i + 1 for i, a in enumerate(AA)}          # 0 = unknown / padding
SS8_TO_3 = {'H': 0, 'G': 0, 'I': 0, 'E': 1, 'B': 1, 'T': 2, 'S': 2, 'C': 2, '-': 2, ' ': 2, 'L': 2, 'P': 2}

import re
_NUM = re.compile(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?')
def floats(s):
    # robust to '1.0 0.0 ...', '[1.0, 0.0, ...]', '1,0,1', ints, NaN, etc.
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.zeros(0, dtype=np.float32)
    return np.array([float(x) for x in _NUM.findall(str(s))], dtype=np.float32)

def load_csv(path, name):
    df = pd.read_csv(path)
    print(f'{name}: {len(df)} rows, columns = {list(df.columns)}')
    recs, skipped, agree, total, warned = [], 0, 0, 0, {}
    for i, r in df.iterrows():
        seq = str(r['input']).strip().upper()
        ss8 = str(r['dssp8']).strip() if 'dssp8' in df.columns else None
        ss3 = str(r['dssp3']).strip() if 'dssp3' in df.columns else None
        L = len(seq)
        if ss8 is not None and len(ss8) == L:
            y = np.array([SS8_TO_3.get(c, -1) for c in ss8], dtype=np.int64)
        elif ss3 is not None and len(ss3) == L:
            y = np.array([{'H': 0, 'E': 1, 'C': 2}.get(c, -1) for c in ss3], dtype=np.int64)
        else:
            skipped += 1; continue
        if ss3 is not None and len(ss3) == L:
            y3 = np.array([{'H': 0, 'E': 1, 'C': 2}.get(c, -1) for c in ss3])
            agree += int(np.sum(y3 == y)); total += L
        m = y >= 0
        for col in ['disorder', 'cb513_mask']:          # 1.0 = residue is evaluated
            if col in df.columns:
                v = floats(r[col])
                if len(v) == L: m &= v > 0.5
                elif len(v) > 0 and not warned.get(col):
                    print(f'   WARNING: {col} length {len(v)} != sequence length {L} (row {i}); raw start: {str(r[col])[:60]!r}')
                    warned[col] = True
        recs.append(dict(name=f'{name}_{i}', seq=seq, y=y, m=m,
                         tok=np.array([AA_IX.get(a, 0) for a in seq], dtype=np.int64)))
    frac = np.mean(np.concatenate([r['m'] for r in recs]))
    print(f'   kept {len(recs)}, skipped {skipped}, evaluated-residue fraction = {frac:.3f}' +
          (f', dssp8->3 vs dssp3 agreement = {agree/total:.4f}' if total else ''))
    if frac < 0.5: print('   WARNING: mask keeps <50% of residues - check mask semantics!')
    return recs

DATA = {k: load_csv(f'/content/data/{f}', k) for k, f in FILES.items()}

In [ ]:
# ==== 4. Homology-leakage filter (MMseqs2): remove training chains similar to ANY test chain ====
LEAK = {'enabled': FILTER_LEAKAGE}
if FILTER_LEAKAGE:
    if not os.path.exists('/content/mmseqs/bin/mmseqs'):
        assert os.system('wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /content/mmseqs.tgz && tar xzf /content/mmseqs.tgz -C /content') == 0
    with open('/content/train.fa', 'w') as f:
        for r in DATA['TRAIN']: f.write(f">{r['name']}\n{r['seq']}\n")
    with open('/content/test.fa', 'w') as f:
        for t in TEST_SETS:
            for r in DATA[t]: f.write(f">{r['name']}\n{r['seq']}\n")
    cmd = (f"/content/mmseqs/bin/mmseqs easy-search /content/test.fa /content/train.fa /content/hits.m8 /content/mmtmp "
           f"--min-seq-id {LEAK_MIN_ID} -c {LEAK_MIN_COV} --cov-mode 0 -s 7.5 -e 1e-3 "
           f"--format-output query,target,fident,alnlen,qcov,tcov,evalue --threads {os.cpu_count()} -v 1")
    print(cmd); assert os.system(cmd) == 0
    cols = ['query', 'target', 'fident', 'alnlen', 'qcov', 'tcov', 'evalue']
    hits = (pd.read_csv('/content/hits.m8', sep='\t', header=None, names=cols)
            if os.path.getsize('/content/hits.m8') > 0 else pd.DataFrame(columns=cols))
    hits = hits[(hits.fident >= LEAK_MIN_ID)]
    hits.to_csv(f'{OUT}/leakage_hits.csv', index=False)
    bad = set(hits.target)
    LEAK.update(min_identity=LEAK_MIN_ID, min_coverage=LEAK_MIN_COV, n_train_before=len(DATA['TRAIN']),
                n_removed=len(bad), per_test_set={t: int(hits['query'].str.startswith(t + '_').sum()) for t in TEST_SETS})
    DATA['TRAIN'] = [r for r in DATA['TRAIN'] if r['name'] not in bad]
    LEAK['n_train_after'] = len(DATA['TRAIN'])
print(LEAK)
json.dump(LEAK, open(f'{OUT}/leakage_filter.json', 'w'), indent=2)

In [ ]:
# ==== 5. Train / validation split + dataset statistics ====
rng = np.random.RandomState(SPLIT_SEED)
perm = rng.permutation(len(DATA['TRAIN']))
n_val = int(round(VAL_FRACTION * len(perm)))
VAL = [DATA['TRAIN'][i] for i in perm[:n_val]]
TRAIN = [DATA['TRAIN'][i] for i in perm[n_val:]]
if QUICK_TEST:
    TRAIN, VAL = TRAIN[:400], VAL[:100]
    for t in TEST_SETS: DATA[t] = DATA[t][:40]
TESTS = {t: DATA[t] for t in TEST_SETS}

def stats_of(recs):
    y = np.concatenate([r['y'][r['m']] for r in recs])
    return dict(n_chains=len(recs), n_residues=int(sum(len(r['seq']) for r in recs)), n_evaluated_residues=int(len(y)),
                frac_H=float(np.mean(y == 0)), frac_E=float(np.mean(y == 1)), frac_C=float(np.mean(y == 2)),
                min_len=int(min(len(r['seq']) for r in recs)), max_len=int(max(len(r['seq']) for r in recs)))
DSTATS = {'TRAIN': stats_of(TRAIN), 'VAL': stats_of(VAL), **{t: stats_of(v) for t, v in TESTS.items()}}
print(pd.DataFrame(DSTATS).T)
json.dump(DSTATS, open(f'{OUT}/dataset_stats.json', 'w'), indent=2)

In [ ]:
# ==== 6. Metrics (all computed from real predictions) ====
import numpy as np
from scipy import stats

def q3(t, p):
    t = np.asarray(t); p = np.asarray(p)
    return 100.0 * float(np.mean(t == p)) if len(t) else float('nan')

def per_class(t, p):
    t = np.asarray(t); p = np.asarray(p); out = {}
    f1s = []
    for k, name in enumerate(['H', 'E', 'C']):
        tp = np.sum((t == k) & (p == k)); fp = np.sum((t != k) & (p == k)); fn = np.sum((t == k) & (p != k))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        out[f'Prec_{name}'] = 100 * pr; out[f'Rec_{name}'] = 100 * rc; out[f'F1_{name}'] = 100 * f1
        f1s.append(f1)
    out['Macro_F1'] = 100 * float(np.mean(f1s))
    return out

def confusion(t, p):
    m = np.zeros((3, 3), dtype=np.int64)
    np.add.at(m, (np.asarray(t), np.asarray(p)), 1)
    return m  # rows = true H,E,C ; cols = predicted H,E,C

def _segments(a):
    segs, start = [], 0
    for i in range(1, len(a) + 1):
        if i == len(a) or a[i] != a[start]:
            segs.append((int(a[start]), start, i - 1)); start = i
    return segs

def sov99_protein(t, p):
    """SOV'99 (Zemla et al. 1999) for one protein. Returns percent, or nan if empty."""
    t = np.asarray(t); p = np.asarray(p)
    if len(t) == 0:
        return float('nan')
    so, sp = _segments(t), _segments(p)
    num, N = 0.0, 0
    for (state, a1, b1) in so:
        l1 = b1 - a1 + 1
        overlapped = False
        for (s2, a2, b2) in sp:
            if a2 > b1:
                break
            if s2 != state or b2 < a1:
                continue
            overlapped = True
            l2 = b2 - a2 + 1
            minov = min(b1, b2) - max(a1, a2) + 1
            maxov = max(b1, b2) - min(a1, a2) + 1
            delta = min(maxov - minov, minov, l1 // 2, l2 // 2)
            num += (minov + delta) / maxov * l1
            N += l1                      # N(i): l1 counted once per overlapping pair
        if not overlapped:
            N += l1                      # ... and once for each non-overlapping observed segment
    return 100.0 * num / N if N else float('nan')

def beta_coil_error(t, p):
    """Bidirectional E<->C confusion: (E predicted as C + C predicted as E) / (#true E + #true C)."""
    t = np.asarray(t); p = np.asarray(p)
    e, c = (t == 1), (t == 2)
    e2c = np.sum(e & (p == 2)); c2e = np.sum(c & (p == 1))
    return {'E_to_C': 100 * e2c / max(1, e.sum()),
            'C_to_E': 100 * c2e / max(1, c.sum()),
            'BetaCoil_Err': 100 * (e2c + c2e) / max(1, e.sum() + c.sum())}

def evaluate(per_protein_true, per_protein_pred):
    """per_protein_*: lists of 1-D int arrays restricted to evaluated (masked-in) residues."""
    t = np.concatenate(per_protein_true); p = np.concatenate(per_protein_pred)
    sovs = [sov99_protein(a, b) for a, b in zip(per_protein_true, per_protein_pred) if len(a)]
    res = {'Q3': q3(t, p), 'SOV99': float(np.nanmean(sovs)), 'n_residues': int(len(t)),
           'n_proteins': int(sum(1 for a in per_protein_true if len(a)))}
    res.update(per_class(t, p)); res.update(beta_coil_error(t, p))
    return res

def mcnemar(t, pa, pb):
    ca, cb = (pa == t), (pb == t)
    b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb))
    if b + c == 0:
        return {'chi2': 0.0, 'p': 1.0, 'b_A_only': b, 'c_B_only': c, 'N': int(len(t))}
    chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)          # with continuity correction
    return {'chi2': float(chi2), 'p': float(stats.chi2.sf(chi2, 1)), 'b_A_only': b, 'c_B_only': c, 'N': int(len(t))}

def wilcoxon_proteins(qa, qb):
    r = stats.wilcoxon(qa, qb, alternative='two-sided', zero_method='wilcox')
    return {'W': float(r.statistic), 'p': float(r.pvalue), 'n_proteins': int(len(qa))}

def bootstrap_pooled_q3(corr_a, corr_b, n_res, B=10000, seed=42):
    """Paired bootstrap over whole proteins of the pooled-Q3 difference (A - B), in percentage points.
    corr_a/corr_b: per-protein number of correct residues; n_res: per-protein residue count."""
    corr_a = np.asarray(corr_a, float); corr_b = np.asarray(corr_b, float); n_res = np.asarray(n_res, float)
    rng = np.random.RandomState(seed); n = len(n_res)
    obs = 100 * (corr_a.sum() - corr_b.sum()) / n_res.sum()
    idx = rng.randint(0, n, size=(B, n))
    d = 100 * (corr_a[idx].sum(1) - corr_b[idx].sum(1)) / n_res[idx].sum(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    p_two = 2 * min(np.mean(d <= 0), np.mean(d >= 0))
    return {'delta_Q3': float(obs), 'ci_low': float(lo), 'ci_high': float(hi),
            'p_boot': float(max(p_two, 1.0 / B)), 'p_is_lower_bound': bool(p_two < 1.0 / B), 'B': B}

In [ ]:
# ==== 7. Models ====
def one_hot_batch(recs, device):
    L = max(len(r['seq']) for r in recs); B = len(recs)
    tok = torch.zeros(B, L, dtype=torch.long)           # 0 = unknown/padding
    y = torch.full((B, L), -100, dtype=torch.long)
    lm = torch.zeros(B, L, dtype=torch.bool)            # residue present
    for i, r in enumerate(recs):
        n = len(r['seq']); tok[i, :n] = torch.from_numpy(r['tok']); lm[i, :n] = True
        yy = torch.from_numpy(r['y'].copy()); yy[~torch.from_numpy(r['m'])] = -100; y[i, :n] = yy
    x = F.one_hot(tok, 21).float()
    return x.to(device), lm.to(device), y.to(device)

class WindowMLP(nn.Module):
    # sliding window W=17 of one-hot(21) -> 357 -> 128 -> 64 -> 3 ; Linear layers before BatchNorm have no bias
    def __init__(self, W=17, d=21, h=128, p=0.2):
        super().__init__(); self.W, self.half = W, W // 2
        self.net = nn.Sequential(nn.Linear(W * d, h, bias=False), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h, h // 2, bias=False), nn.BatchNorm1d(h // 2), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h // 2, 3))
    def forward(self, x, lm):
        B, L, d = x.shape
        xp = F.pad(x, (0, 0, self.half, self.half)); xp[:, :self.half, 0] = 1; xp[:, -self.half:, 0] = 1
        win = xp.unfold(1, self.W, 1).permute(0, 1, 3, 2).reshape(B, L, self.W * d)
        out = x.new_zeros(B, L, 3); out[lm] = self.net(win[lm]); return out

class HybridCNNBiLSTM(nn.Module):
    def __init__(self, d=21, conv=128, hid=128, p=0.3):
        super().__init__(); a = conv // 3; c = conv - 2 * a
        self.c3, self.c5, self.c7 = nn.Conv1d(d, a, 3, padding=1), nn.Conv1d(d, a, 5, padding=2), nn.Conv1d(d, c, 7, padding=3)
        self.bn = nn.BatchNorm1d(conv); self.sdrop = nn.Dropout1d(p)          # spatial (channel) dropout
        self.lstm = nn.LSTM(conv, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=p)
        self.ln = nn.LayerNorm(2 * hid)
        self.head = nn.Sequential(nn.Linear(2 * hid, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 3))
    def forward(self, x, lm):
        h = x.transpose(1, 2); h = torch.cat([self.c3(h), self.c5(h), self.c7(h)], 1)
        h = self.sdrop(F.relu(self.bn(h))).transpose(1, 2)
        lengths = lm.sum(1).cpu()
        pk = nn.utils.rnn.pack_padded_sequence(h, lengths, batch_first=True, enforce_sorted=False)
        o, _ = self.lstm(pk); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=x.shape[1])
        return self.head(self.ln(o))

def make_head(kind, d):
    if kind == 'linear': return nn.Linear(d, 3)
    return nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))

def n_params(m, trainable_only=False):
    return int(sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only)))
def weight_mb(*ms):
    return sum(p.numel() * p.element_size() for m in ms for p in m.parameters()) / 2**20

for nm, m in [('WindowMLP', WindowMLP()), ('Hybrid', HybridCNNBiLSTM()), ('Linear320', make_head('linear', 320)),
              ('MLP320', make_head('mlp', 320)), ('MLP480', make_head('mlp', 480))]:
    print(f'{nm:10s} params = {n_params(m):,}')

In [ ]:
# ==== 8. Training utilities ====
def length_batches(recs, bs, shuffle, rng=None):
    idx = np.argsort([len(r['seq']) for r in recs])
    groups = [idx[i:i + bs] for i in range(0, len(idx), bs)]
    if shuffle: rng.shuffle(groups)
    return [[recs[j] for j in g] for g in groups]

def make_opt(cfg, params):
    return (torch.optim.AdamW if cfg['opt'] == 'AdamW' else torch.optim.Adam)(params, lr=cfg['lr'], weight_decay=cfg['wd'])

def train_seq_model(name, seed):
    cfg = CFG[name]; set_seed(seed); rng = np.random.RandomState(seed)
    model = (WindowMLP() if name == 'WindowMLP' else HybridCNNBiLSTM()).to(dev)
    opt = make_opt(cfg, model.parameters()); crit = nn.CrossEntropyLoss(ignore_index=-100, reduction='sum')
    hist = {'train': [], 'val': []}; best, best_state, bad = 1e9, None, 0
    for ep in range(cfg['max_epochs']):
        model.train(); tl, tn = 0.0, 0
        for b in length_batches(TRAIN, cfg['batch'], True, rng):
            x, lm, y = one_hot_batch(b, dev)
            loss = crit(model(x, lm).reshape(-1, 3), y.reshape(-1)); n = int((y != -100).sum())
            opt.zero_grad(); (loss / max(n, 1)).backward(); opt.step(); tl += loss.item(); tn += n
        model.eval(); vl, vn = 0.0, 0
        with torch.no_grad():
            for b in length_batches(VAL, 64, False):
                x, lm, y = one_hot_batch(b, dev)
                vl += crit(model(x, lm).reshape(-1, 3), y.reshape(-1)).item(); vn += int((y != -100).sum())
        hist['train'].append(tl / tn); hist['val'].append(vl / vn)
        print(f'  [{name} seed {seed}] epoch {ep+1}: train {tl/tn:.4f}  val {vl/vn:.4f}')
        if vl / vn < best - 1e-4: best, best_state, bad = vl / vn, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= cfg['patience']: break
    model.load_state_dict(best_state); model.eval()
    hist['best_epoch'] = int(np.argmin(hist['val']) + 1)
    return model, hist

@torch.no_grad()
def predict_seq_model(model, recs):
    res = {}
    for b in length_batches(recs, 32, False):
        x, lm, y = one_hot_batch(b, dev); pr = model(x, lm).argmax(-1).cpu().numpy()
        for i, r in enumerate(b):
            n = len(r['seq']); m = r['m']; res[r['name']] = (r['y'][m], pr[i, :n][m])
    return [res[r['name']][0] for r in recs], [res[r['name']][1] for r in recs]

In [ ]:
# ==== 9. ESM-2 frozen embeddings (computed once per backbone; backbone is never trained) ====
from transformers import AutoTokenizer, EsmModel
MAXC = 1022
def clean(s): return ''.join(c if c in AA else 'X' for c in s)

def load_backbone(name, device):
    tok = AutoTokenizer.from_pretrained(name)
    bb = EsmModel.from_pretrained(name, add_pooling_layer=False).to(device).eval()
    for p in bb.parameters(): p.requires_grad = False
    return tok, bb

@torch.no_grad()
def embed_seqs(tok, bb, seqs, device, max_tokens=12000):
    # returns list of [L, d] float16 CPU tensors; sequences > 1022 are embedded in consecutive 1022-residue chunks
    pieces = []
    for si, s in enumerate(seqs):
        s = clean(s)
        for st in range(0, len(s), MAXC): pieces.append((si, st, s[st:st + MAXC]))
    order = sorted(range(len(pieces)), key=lambda k: len(pieces[k][2]))
    out = {}; i = 0
    while i < len(order):
        L = len(pieces[order[i]][2]); bs = max(1, max_tokens // (L + 2)); j = i
        batch = []
        while j < len(order) and len(batch) < bs and len(pieces[order[j]][2]) <= L * 1.0 + 64: batch.append(order[j]); j += 1
        enc = tok([pieces[k][2] for k in batch], return_tensors='pt', padding=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
            h = bb(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device)).last_hidden_state
        am = enc['attention_mask'].sum(1)
        for bi, k in enumerate(batch):
            n = len(pieces[k][2]); assert int(am[bi]) == n + 2, 'tokenizer did not give one token per residue'
            out[k] = h[bi, 1:n + 1].half().cpu()
        i = j
    by_seq = {}
    for k, (si, st, _) in enumerate(pieces): by_seq.setdefault(si, []).append((st, k))
    return [torch.cat([out[k] for st, k in sorted(by_seq[si])], 0) for si in range(len(seqs))]

def residue_tensor(recs, embs):
    X = torch.cat([e[torch.from_numpy(r['m'])] for r, e in zip(recs, embs)], 0)
    y = torch.from_numpy(np.concatenate([r['y'][r['m']] for r in recs]))
    return X.to(dev), y.to(dev)

def train_head(name, seed, Xtr, ytr, Xva, yva, d):
    cfg = CFG[name]; set_seed(seed)
    head = make_head(cfg['head'], d).to(dev); opt = make_opt(cfg, head.parameters())
    crit = nn.CrossEntropyLoss(reduction='sum'); g = torch.Generator(device=dev); g.manual_seed(seed)
    hist = {'train': [], 'val': []}; best, best_state, bad = 1e9, None, 0
    for ep in range(cfg['max_epochs']):
        head.train(); perm = torch.randperm(len(ytr), device=dev, generator=g); tl = 0.0
        for i in range(0, len(perm), cfg['batch']):
            ix = perm[i:i + cfg['batch']]; loss = crit(head(Xtr[ix].float()), ytr[ix])
            opt.zero_grad(); (loss / len(ix)).backward(); opt.step(); tl += loss.item()
        head.eval(); vl = 0.0
        with torch.no_grad():
            for i in range(0, len(yva), 65536): vl += crit(head(Xva[i:i + 65536].float()), yva[i:i + 65536]).item()
        hist['train'].append(tl / len(ytr)); hist['val'].append(vl / len(yva))
        print(f'  [{name} seed {seed}] epoch {ep+1}: train {tl/len(ytr):.4f}  val {vl/len(yva):.4f}')
        if vl / len(yva) < best - 1e-4: best, best_state, bad = vl / len(yva), copy.deepcopy(head.state_dict()), 0
        else:
            bad += 1
            if bad >= cfg['patience']: break
    head.load_state_dict(best_state); head.eval(); hist['best_epoch'] = int(np.argmin(hist['val']) + 1)
    return head, hist

@torch.no_grad()
def predict_head(head, recs, embs):
    T, P = [], []
    for r, e in zip(recs, embs):
        p = head(e.to(dev).float()).argmax(-1).cpu().numpy()
        T.append(r['y'][r['m']]); P.append(p[r['m']])
    return T, P

In [ ]:
# ==== 10. Throughput / latency / memory (batch size 1, real CB513 sequences, end-to-end from the string) ====
BENCH = [r for r in TESTS['CB513']]
def _sync(d):
    if d.type == 'cuda': torch.cuda.synchronize()

@torch.no_grad()
def bench(fn, device, n):
    seqs = BENCH[:n]
    for r in seqs[:3]: fn(r)                                  # warm-up
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    _sync(device); t0 = time.perf_counter()
    for r in seqs: fn(r)
    _sync(device); dt = time.perf_counter() - t0
    res = {'seq_per_s': len(seqs) / dt, 'ms_per_seq': 1000 * dt / len(seqs), 'n_seqs': len(seqs),
           'mean_len': float(np.mean([len(r['seq']) for r in seqs]))}
    if device.type == 'cuda': res['peak_gpu_mem_MB'] = torch.cuda.max_memory_allocated() / 2**20
    return res

def seq_model_fn(model, device):
    def f(r):
        x, lm, _ = one_hot_batch([r], device); return model(x, lm).argmax(-1)
    return f

def esm_fn(tok, bb, head, device):
    def f(r):
        s = clean(r['seq'])[:MAXC]; enc = tok([s], return_tensors='pt')
        h = bb(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device)).last_hidden_state[0, 1:-1]
        return head(h).argmax(-1)
    return f

def bench_all(make_fn, modules, do_cpu):
    out = {}
    g = bench(make_fn(dev), dev, N_BENCH_GPU)
    out.update({'GPU_seq_per_s': g['seq_per_s'], 'GPU_ms_per_seq': g['ms_per_seq'], 'GPU_peak_mem_MB': g['peak_gpu_mem_MB']})
    if do_cpu:
        torch.set_num_threads(CPU_THREADS)
        for m in modules: m.cpu()
        c = bench(make_fn(torch.device('cpu')), torch.device('cpu'), N_BENCH_CPU)
        for m in modules: m.to(dev)
        out.update({'CPU_seq_per_s': c['seq_per_s'], 'CPU_ms_per_seq': c['ms_per_seq']})
    out['Weights_MB_fp32'] = weight_mb(*modules)
    return out

In [ ]:
# ==== 11. RUN EVERYTHING ====
RUNS, CURVES, PREDS = [], {}, {}
def record(name, seed, evals, eff, tot, trn, hist):
    row = {'model': name, 'seed': seed, 'Total_Params': tot, 'Trainable_Params': trn, 'best_epoch': hist['best_epoch'],
           'epochs_run': len(hist['train'])}
    for t, e in evals.items():
        for k in ['Q3', 'SOV99', 'Macro_F1', 'F1_H', 'F1_E', 'F1_C', 'BetaCoil_Err', 'E_to_C', 'C_to_E', 'n_residues', 'n_proteins']:
            row[f'{t}_{k}'] = e[k]
    row.update(eff); RUNS.append(row); CURVES[f'{name}|{seed}'] = hist
    pd.DataFrame(RUNS).to_csv(f'{OUT}/all_seed_runs.csv', index=False)
    json.dump(CURVES, open(f'{OUT}/loss_curves.json', 'w'))
    print(f"==> {name} seed {seed}: " + ', '.join(f"{t} Q3={evals[t]['Q3']:.2f}" for t in evals))

# --- non-pretrained models ---
for name in ['WindowMLP', 'Hybrid 1D-CNN-BiLSTM']:
    for seed in SEEDS:
        model, hist = train_seq_model(name, seed)
        evals = {}
        for t, recs in TESTS.items():
            T, P = predict_seq_model(model, recs); evals[t] = evaluate(T, P); PREDS[(name, seed, t)] = (T, P)
        eff = bench_all(lambda d: seq_model_fn(model, d), [model], do_cpu=(seed == SEEDS[0]))
        record(name, seed, evals, eff, n_params(model), n_params(model, True), hist)
        del model; torch.cuda.empty_cache()

# --- frozen ESM-2 backbones ---
for bname in ['facebook/esm2_t6_8M_UR50D', 'facebook/esm2_t12_35M_UR50D']:
    tok, bb = load_backbone(bname, dev); d = bb.config.hidden_size
    print(f'{bname}: hidden={d}, layers={bb.config.num_hidden_layers}, heads={bb.config.num_attention_heads}, params={n_params(bb):,}')
    t0 = time.time()
    E_tr = embed_seqs(tok, bb, [r['seq'] for r in TRAIN], dev); E_va = embed_seqs(tok, bb, [r['seq'] for r in VAL], dev)
    E_te = {t: embed_seqs(tok, bb, [r['seq'] for r in recs], dev) for t, recs in TESTS.items()}
    print(f'  embeddings computed in {time.time()-t0:.0f} s')
    Xtr, ytr = residue_tensor(TRAIN, E_tr); Xva, yva = residue_tensor(VAL, E_va); del E_tr, E_va
    for name in [n for n, c in CFG.items() if c.get('backbone') == bname]:
        for seed in SEEDS:
            head, hist = train_head(name, seed, Xtr, ytr, Xva, yva, d)
            evals = {}
            for t, recs in TESTS.items():
                T, P = predict_head(head, recs, E_te[t]); evals[t] = evaluate(T, P); PREDS[(name, seed, t)] = (T, P)
            eff = bench_all(lambda dv: esm_fn(tok, bb, head, dv), [bb, head], do_cpu=(seed == SEEDS[0]))
            record(name, seed, evals, eff, n_params(bb) + n_params(head), n_params(head, True), hist)
    del Xtr, ytr, Xva, yva, E_te, bb; torch.cuda.empty_cache()

In [ ]:
# ==== 12. Summary table (mean ± SD over seeds) ====
runs = pd.DataFrame(RUNS)
metric_cols = [c for c in runs.columns if c not in ('model', 'seed')]
agg = runs.groupby('model', sort=False)[metric_cols].agg(['mean', 'std'])
agg.columns = [f'{a}_{b}' for a, b in agg.columns]
agg.to_csv(f'{OUT}/summary_metrics.csv')
show = ['CB513_Q3', 'CB513_SOV99', 'TS115_Q3', 'CASP12_Q3', 'CASP13_Q3', 'CASP14_Q3', 'CB513_BetaCoil_Err',
        'GPU_seq_per_s', 'GPU_peak_mem_MB', 'Weights_MB_fp32', 'Total_Params', 'Trainable_Params']
tbl = pd.DataFrame({c: [f"{agg.loc[m, c+'_mean']:.2f} ± {0 if np.isnan(agg.loc[m, c+'_std']) else agg.loc[m, c+'_std']:.2f}" for m in agg.index] for c in show}, index=agg.index)
cpu = runs.dropna(subset=['CPU_seq_per_s']).set_index('model')[['CPU_seq_per_s', 'CPU_ms_per_seq']]
print(tbl.to_string()); print('\nCPU (single run, %d threads):' % CPU_THREADS); print(cpu.to_string())
tbl.to_csv(f'{OUT}/summary_table_pretty.csv')

# per-class + confusion matrices (seed = STATS_SEED, all test sets)
rows = []
for (name, seed, t), (T, P) in PREDS.items():
    if seed != STATS_SEED: continue
    cm = confusion(np.concatenate(T), np.concatenate(P))
    rows.append({'model': name, 'test_set': t, **{f'cm_{a}{b}': int(cm[i, j]) for i, a in enumerate('HEC') for j, b in enumerate('HEC')}})
pd.DataFrame(rows).to_csv(f'{OUT}/confusion_matrices.csv', index=False)

In [ ]:
# ==== 13. Statistical tests on real predictions ====
def paired(a_name, b_name, t=STATS_TEST_SET, seed=STATS_SEED):
    Ta, Pa = PREDS[(a_name, seed, t)]; Tb, Pb = PREDS[(b_name, seed, t)]
    keep = [i for i in range(len(Ta)) if len(Ta[i]) > 0]
    tt = np.concatenate([Ta[i] for i in keep]); pa = np.concatenate([Pa[i] for i in keep]); pb = np.concatenate([Pb[i] for i in keep])
    ca = np.array([np.sum(Pa[i] == Ta[i]) for i in keep]); cb = np.array([np.sum(Pb[i] == Tb[i]) for i in keep])
    nr = np.array([len(Ta[i]) for i in keep])
    mc = mcnemar(tt, pa, pb); wx = wilcoxon_proteins(100 * ca / nr, 100 * cb / nr); bs = bootstrap_pooled_q3(ca, cb, nr, B=BOOT_B)
    return {'Comparison': f'{a_name} vs {b_name}', 'test_set': t, 'seed': seed,
            'McNemar_N_residues': mc['N'], 'McNemar_b_A_only': mc['b_A_only'], 'McNemar_c_B_only': mc['c_B_only'],
            'McNemar_chi2': mc['chi2'], 'McNemar_p': mc['p'], 'Wilcoxon_n_proteins': wx['n_proteins'], 'Wilcoxon_W': wx['W'],
            'Wilcoxon_p': wx['p'], 'Delta_Q3_pooled': bs['delta_Q3'], 'Boot_CI_low': bs['ci_low'], 'Boot_CI_high': bs['ci_high'],
            'Boot_p': bs['p_boot'], 'Boot_p_is_upper_bound_1_over_B': bs['p_is_lower_bound'], 'Boot_B': bs['B'],
            'Mean_per_protein_delta_Q3': float(np.mean(100 * ca / nr - 100 * cb / nr))}
ST = pd.DataFrame([paired('ESM-2 (8M) MLP Head', 'Hybrid 1D-CNN-BiLSTM'),
                   paired('ESM-2 (35M) MLP Head', 'ESM-2 (8M) MLP Head'),
                   paired('ESM-2 (8M) MLP Head', 'ESM-2 (8M) Linear Probe')])
ST.to_csv(f'{OUT}/statistical_tests.csv', index=False); print(ST.T.to_string())
np.savez_compressed(f'{OUT}/predictions.npz', **{f'{n}|{s}|{t}|{k}': np.concatenate(v[i]) for (n, s, t), v in PREDS.items() for i, k in enumerate(['true', 'pred'])})

In [ ]:
# ==== 14. Loss-curve figure (real per-epoch losses, seed = STATS_SEED) + download ====
import matplotlib.pyplot as plt, shutil
fig, ax = plt.subplots(figsize=(8.6, 5.2), dpi=200)
for i, name in enumerate(CFG):
    h = CURVES[f'{name}|{STATS_SEED}']; ep = range(1, len(h['train']) + 1); c = f'C{i}'
    ax.plot(ep, h['train'], '-o', ms=3, color=c, label=f'{name} (train)'); ax.plot(ep, h['val'], '--s', ms=3, color=c, label=f'{name} (val)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss'); ax.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig(f'{OUT}/loss_curves_real.png'); plt.show()
shutil.make_archive('/content/results_real', 'zip', OUT)
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive'); shutil.copy('/content/results_real.zip', '/content/drive/MyDrive/')
from google.colab import files; files.download('/content/results_real.zip')
print('DONE. QUICK_TEST =', QUICK_TEST, '-> do not use QUICK_TEST results in the paper.' if QUICK_TEST else '')